# Specifications

In [1]:
import SLiCAP as sl
import sympy as sp
sl.initProject('Hearing loop', notebook=True)

Compiling library: SLiCAP.lib.
Compiling library: SLiCAPmodels.lib.


## Feedback model
Below, the feedback concept of the hearing loop receive coil amplifier with the source, the load, and the DIN A noise weighting function.

**Tasks**
1. Design at least two more negative-feedback amplifier configurations
2. Discuss the pros and cons of these configurations with respect to the one given below.

In [2]:
cir = sl.makeCircuit("kicad/A1/feedbackConcept.kicad_sch")
sl.img2html("feedbackConcept.svg", 900)

Creating netlist of kicad/A1/feedbackConcept.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/feedbackConcept.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/feedbackConcept.svg'.
Done.
Checking netlist: cir/feedbackConcept.cir


## Required budgets for the design of the feedback network

In [3]:
# Relative budget for the current through the feedback network to the maximum total current
B_i1 = 0.1
# Relative budget for the contribution of the source, its termination, and the feedback network to the total weighted output noise
B_n1 = 0.4

## Component and specifications
We will import the specifications and assign parameter values to circuit elements. 

In [4]:
specs = sl.csv2specs('A1specs.csv')

The integrator capacitance $C_i$ will be selected such that the integration time constant is $\tau_i$:
\begin{equation}
C_i = \frac{\tau_i}{R_f}
\end{equation}
The value of the bias resistor will be such that the high-pass cut-off frequency equals $f_\mathrm{min}$:
\begin{equation}
R_b=\frac{1}{2\pi f_\mathrm{min} C_i}
\end{equation}
These expressions will be added to the specifications:

In [5]:
Ci = sp.sympify("tau_i/R_f")
Rb = sp.sympify("1/(2*pi*f_min*C_i)")

specs.append(
    sl.specItem(
        "C_i",
        description = "Integrator capacitance",
        value       = Ci,
        units       = "F",
        specType    = "design",
    )
)
specs.append(
    sl.specItem(
        "R_b",
        description = "Bias resistance",
        value       = Rb,
        units       = "Omega",
        specType    = "design",
    )
)
sl.specs2circuit(specs, cir)

We now have have to determine $R_f$, which is the only unknown parameter in the circuit elements:

In [6]:
sl.params2html(cir)

Name,Symbolic,Numeric
$C_{L}$,$10\cdot 10^{-12}$,$10\cdot 10^{-12}$
$C_{i}$,$\frac{\tau_{i}}{R_{f}}$,$\frac{1.606 \cdot 10^{-5}}{R_{f}}$
$C_{s}$,$9.382\cdot 10^{-12}$,$9.382\cdot 10^{-12}$
$DIN_{A}$,$\frac{1.872 \cdot 10^{8} f^{4}}{\left(\left(f^{2} + 1.16 \cdot 10^{4}\right) \left(f^{2} + 5.445 \cdot 10^{5}\right)\right)^{0.5} \left(f^{2} + 424.4\right) \left(f^{2} + 1.487 \cdot 10^{8}\right)}$,$\frac{1.872 \cdot 10^{8} f^{4}}{\left(\left(f^{2} + 1.16 \cdot 10^{4}\right) \left(f^{2} + 5.445 \cdot 10^{5}\right)\right)^{0.5} \left(f^{2} + 424.4\right) \left(f^{2} + 1.487 \cdot 10^{8}\right)}$
$L_{s}$,$0.12$,$0.12$
$P_{max}$,$1000\cdot 10^{-6}$,$1000\cdot 10^{-6}$
$R_{b}$,$\frac{0.5}{\pi C_{i} f_{min}}$,$16.51 R_{f}$
$R_{s}$,$875$,$875$
$R_{t}$,$79.97\cdot 10^{3}$,$79.97\cdot 10^{3}$
$T$,$300$,$300$


## Minimum value of $R_f$
The lower limit show-stopper value of $R_f$ follows from the maximum power dissipation and the maximum power supply voltage.
In fact, we can define a maximum current budget $B_\mathrm{i_1}$ for it as a relative part of the maximum power supply current.
\begin{equation}
I_\mathrm{R_3} = B_\mathrm{i_1}\frac{P_{max}}{V_{DD_{max}}}
\end{equation}
Circuit analysis yields:
\begin{equation}
I_\mathrm{R_3} = \frac{Vi_\mathrm{pp}}{R_f}
\end{equation}
We will use sympy to solve this equation after setting this budget:

In [7]:
specs.append(
    sl.specItem(
        "B_i1",
        description = "Relative budget for the current through the feedback network",
        value       = B_i1,
        units       = "",
        specType    = "design",
    )
)
spec_dict = sl.specs2dict(specs)
sl.eqn2html("B_i1", spec_dict["B_i1"].value)

In [8]:
B_i1     = spec_dict["B_i1"].value
P_max    = spec_dict["P_max"].value
V_DD_max = spec_dict["V_DD_max"].value
Vi_pp    = spec_dict["Vi_pp"].value
R_f      = sp.Symbol("R_f")
R_f_min  = sp.solve(B_i1*P_max/V_DD_max - Vi_pp/R_f, R_f)[0]
# Add it to the specifications
specs.append(
    sl.specItem(
        "R_f_min",
        description = "Minimum value of R_f",
        value       = R_f_min,
        units       = "Omega",
        specType    = "design",
    )
)
sl.eqn2html("R_f_min", R_f_min, units="Ohm")

## Maximum value of $R_f$
The maximum value $R_\mathrm{f_{max}}$ of $R_f$ follows from its noise contribution. Let us define a budget $B_\mathrm{n1}$ for the relative contribution of the feedback element plus the source and its termination to the total weighted output noise:

In [9]:
specs.append(
    sl.specItem(
        "B_n1",
        description = "Relative budget for contribution of source, termination and feedback network to the total weighted output noise",
        value       = B_n1,
        units       = "",
        specType    = "design",
    )
)
spec_dict = sl.specs2dict(specs)
S_onoise  = sl.doNoise(cir, pardefs="circuit", numeric=True, detector="V_noise")
sl.eqn2html("B_n1", spec_dict["B_n1"].value)

In [10]:
f_min = spec_dict["f_min"].value
f_max = spec_dict["f_max"].value
# The function "integrate_monomial_coeffs()" is only defined for two variables, so we use one dummy variable "x".
x     = sp.Symbol("x")   
Varn1 = sl.integrate_monomial_coeffs(S_onoise.onoise, [R_f, x], sl.ini.frequency, f_min, f_max, doit=True)
sl.eqn2html("var_n1", Varn1, units="V**2")

The maximum value of $R_f$ is found by solving it from equating the above function with the budget for the contribution to weighted the squared output noise.

In [11]:
B_n1  = spec_dict["B_n1"].value
VarB1 = (spec_dict["V_onoise"].value * B_n1)**2
R_f_max = sp.solve(Varn1 - VarB1, R_f)[0]
sl.eqn2html("R_f_max", R_f_max, units= "Ohm")

In [12]:
specs.append(
    sl.specItem(
        "R_f_max",
        description = "Maximum value of R_f",
        value       = R_f_max,
        units       = "Omega",
        specType    = "design",
    )
)
spec_dict = sl.specs2dict(specs)

# Component selection
Let us select a value for $R_f$:

In [13]:
Rf = 0
while Rf < spec_dict["R_f_min"].value or Rf > spec_dict["R_f_max"].value:
    Rf = float(input("Please enter a value for R_f: {} < R_f < {}:\n>>> ".format(str(sp.N(spec_dict["R_f_min"].value, 3)), 
                                                                               str(sp.N(spec_dict["R_f_max"].value, 3)))))
specs.append(
    sl.specItem(
        "R_f",
        description = "Selected value of R_f",
        value       = Rf,
        units       = "Omega",
        specType    = "design",
    )
)

Please enter a value for R_f: 1.39e+3 < R_f < 7.54e+3:
>>>  7500


In [14]:
sl.specs2html(specs, ["design"])

symbol,description,value,units,$B_{i1}$,Relative budget for the current through the feedback network,$0.1$,,$B_{n1}$,"Relative budget for contribution of source, termination and feedback network to the total weighted output noise",$0.4$,,$C_{i}$,Integrator capacitance,$\frac{\tau_{i}}{R_{f}}$,$\mathrm{F}$,$R_{b}$,Bias resistance,$\frac{0.5}{\pi C_{i} f_{min}}$,$\mathrm{\Omega}$,$R_{f}$,Selected value of R_f,$7.500\cdot 10^{3}$,$\mathrm{\Omega}$,$R_{f max}$,Maximum value of R_f,$7.542\cdot 10^{3}$,$\mathrm{\Omega}$,$R_{f min}$,Minimum value of R_f,$1.393\cdot 10^{3}$,$\mathrm{\Omega}$


# Design verification
## Transfer

In [15]:
sl.specs2circuit(specs, cir)
transfer = sl.doLaplace(cir, pardefs="circuit", numeric=True)
sl.plotSweep("dBmagFBC","dB magnitude transfer feedback concept", transfer, 20, 20000, 200, funcType="dBmag")
sl.img2html("dBmagFBC.svg", 600)

## Noise
We will calculate the realized relative noise budget $Bn_\mathrm{fbc}$ taken by the source and the feedback network:

In [16]:
noiseResult = sl.doNoise(cir, pardefs="circuit", numeric=True, detector="V_noise")
Vn = sl.rmsNoise(noiseResult, "onoise", f_min, f_max)
Bnfbc = Vn/spec_dict["V_onoise"].value
sl.eqn2html("Bn_fbc", Bnfbc)

## Current

In [17]:
I_fb = sl.ini.laplace * spec_dict["Vi_pp"].value/(2*sp.pi*spec_dict["f_fp"].value * Rf)
transfer.laplace = I_fb
transfer.label="current"
sl.plotSweep("magIFBC","Peak-peak current feedback network", transfer, 20, spec_dict["f_fp"].value, 200, yUnits="A", yScale="u")
sl.img2html("magIFBC.svg", 600)

In [18]:
Ipp_ffp = sp.Abs(I_fb.subs(sl.ini.laplace, 2*sp.I*sp.pi*spec_dict["f_fp"].value))
sl.eqn2html("Ipp_ffp", Ipp_ffp, units="A")

The maximum supply current $I_\mathrm{dd}$ is:

In [19]:
Idd = spec_dict["P_max"].value/spec_dict["V_DD_max"].value
sl.eqn2html("I_dd", Idd, units="A")

In [20]:
Bifbc = Ipp_ffp/Idd
sl.eqn2html("Bi_fbc", Bifbc)

In [21]:
sl.specs2csv(specs, "fb_concept.csv")

**Task**

Give design criteria for the feedback components of one other feedback configuration.